# ML Model Optimization Notebook
**Goal:** Take a baseline classifier and systematically improve it using:
1. Preprocessing pipelines
2. Baseline model comparison (cross-validation)
3. Feature selection
4. Hyperparameter tuning (GridSearchCV, RandomizedSearchCV, optional Optuna)
5. Evaluation, learning curves & feature importance
6. Saving the final model

Dataset: Breast Cancer Wisconsin (built into scikit-learn). Swap in your own data in Section 1.

## 1. Setup & Data Loading

In [ ]:
import warnings, time, joblib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score,
                                     GridSearchCV, RandomizedSearchCV, learning_curve)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay)
from scipy.stats import randint, uniform, loguniform

RANDOM_STATE = 42
SCORING = "roc_auc"   # change to "accuracy", "f1", etc.

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
print("Shape:", X.shape)
print("Class balance:\n", y.value_counts(normalize=True).round(3))
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(X_train.shape, X_test.shape)

## 2. Baseline Models (Cross-Validated)
Scaling lives **inside** the pipeline so no information leaks from validation folds.

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "SVC": SVC(probability=True, random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(random_state=RANDOM_STATE),
    "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

rows = []
for name, model in models.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", model)])
    t0 = time.time()
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1)
    rows.append({"model": name, "cv_mean": scores.mean(), "cv_std": scores.std(),
                 "time_s": time.time() - t0})

baseline_df = pd.DataFrame(rows).sort_values("cv_mean", ascending=False).reset_index(drop=True)
baseline_df.round(4)

In [ ]:
plt.figure(figsize=(7,4))
plt.barh(baseline_df["model"], baseline_df["cv_mean"], xerr=baseline_df["cv_std"], color="steelblue")
plt.xlabel(f"CV {SCORING}"); plt.title("Baseline model comparison")
plt.xlim(baseline_df["cv_mean"].min() - 0.02, 1.0); plt.gca().invert_yaxis()
plt.tight_layout(); plt.show()

## 3. Feature Selection
Compare how many features (`k`) are really needed using `SelectKBest`.

In [ ]:
ks = [5, 10, 15, 20, 25, 30]
fs_scores = []
for k in ks:
    pipe = Pipeline([("scaler", StandardScaler()),
                     ("select", SelectKBest(f_classif, k=k)),
                     ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])
    s = cross_val_score(pipe, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1)
    fs_scores.append(s.mean())

plt.plot(ks, fs_scores, marker="o")
plt.xlabel("Number of features (k)"); plt.ylabel(f"CV {SCORING}")
plt.title("Feature selection impact"); plt.grid(alpha=.3); plt.show()
print("Best k:", ks[int(np.argmax(fs_scores))])

## 4. Hyperparameter Tuning
### 4a. GridSearchCV (small, exhaustive space) — SVC

In [ ]:
svc_pipe = Pipeline([("scaler", StandardScaler()), ("clf", SVC(probability=True, random_state=RANDOM_STATE))])
svc_grid = {
    "clf__C": [0.1, 1, 10, 100],
    "clf__gamma": ["scale", 0.001, 0.01, 0.1],
    "clf__kernel": ["rbf", "linear"],
}
t0 = time.time()
grid = GridSearchCV(svc_pipe, svc_grid, cv=cv, scoring=SCORING, n_jobs=-1)
grid.fit(X_train, y_train)
print(f"GridSearch done in {time.time()-t0:.1f}s")
print("Best params:", grid.best_params_)
print("Best CV score:", round(grid.best_score_, 4))

### 4b. RandomizedSearchCV (large space) — Random Forest

In [ ]:
rf_pipe = Pipeline([("scaler", StandardScaler()), ("clf", RandomForestClassifier(random_state=RANDOM_STATE))])
rf_dist = {
    "clf__n_estimators": randint(100, 600),
    "clf__max_depth": [None, 5, 10, 20, 30],
    "clf__min_samples_split": randint(2, 12),
    "clf__min_samples_leaf": randint(1, 6),
    "clf__max_features": ["sqrt", "log2", 0.5],
}
t0 = time.time()
rand = RandomizedSearchCV(rf_pipe, rf_dist, n_iter=40, cv=cv, scoring=SCORING,
                          n_jobs=-1, random_state=RANDOM_STATE)
rand.fit(X_train, y_train)
print(f"RandomizedSearch done in {time.time()-t0:.1f}s")
print("Best params:", rand.best_params_)
print("Best CV score:", round(rand.best_score_, 4))

### 4c. Gradient Boosting + Logistic Regression (Randomized)

In [ ]:
gb_pipe = Pipeline([("scaler", StandardScaler()), ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE))])
gb_dist = {
    "clf__n_estimators": randint(100, 500),
    "clf__learning_rate": loguniform(0.01, 0.3),
    "clf__max_depth": randint(2, 6),
    "clf__subsample": uniform(0.6, 0.4),
}
gb = RandomizedSearchCV(gb_pipe, gb_dist, n_iter=30, cv=cv, scoring=SCORING, n_jobs=-1, random_state=RANDOM_STATE)
gb.fit(X_train, y_train)

lr_pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))])
lr_dist = {"clf__C": loguniform(1e-3, 1e3), "clf__penalty": ["l2"], "clf__solver": ["lbfgs"]}
lr = RandomizedSearchCV(lr_pipe, lr_dist, n_iter=30, cv=cv, scoring=SCORING, n_jobs=-1, random_state=RANDOM_STATE)
lr.fit(X_train, y_train)

print("GB best CV:", round(gb.best_score_, 4), "| LR best CV:", round(lr.best_score_, 4))

### 4d. (Optional) Bayesian Optimization with Optuna
Uncomment the install line if Optuna isn't available. Set `RUN_OPTUNA = True` to run.

In [ ]:
RUN_OPTUNA = False

if RUN_OPTUNA:
    # !pip install optuna
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "max_depth": trial.suggest_int("max_depth", 2, 6),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        }
        pipe = Pipeline([("scaler", StandardScaler()),
                         ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE, **params))])
        return cross_val_score(pipe, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1).mean()

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=40)
    print("Best value:", study.best_value)
    print("Best params:", study.best_params)

## 5. Compare Tuned Models & Pick the Winner

In [ ]:
tuned = {"SVC (grid)": grid, "RandomForest (random)": rand,
         "GradientBoosting (random)": gb, "LogisticRegression (random)": lr}

summary = pd.DataFrame({
    "model": list(tuned.keys()),
    "tuned_cv": [m.best_score_ for m in tuned.values()],
}).sort_values("tuned_cv", ascending=False).reset_index(drop=True)
summary.round(4)

In [ ]:
best_name = summary.loc[0, "model"]
best_model = tuned[best_name].best_estimator_
print("Selected model:", best_name)
print(tuned[best_name].best_params_)

## 6. Final Evaluation on Held-Out Test Set
The test set is touched **only once**, here.

In [ ]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"F1       : {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_proba):.4f}\n")
print(classification_report(y_test, y_pred, target_names=data.target_names))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=data.target_names).plot(ax=ax[0], colorbar=False)
RocCurveDisplay.from_predictions(y_test, y_proba, ax=ax[1])
ax[0].set_title("Confusion matrix"); ax[1].set_title("ROC curve")
plt.tight_layout(); plt.show()

## 7. Learning Curve (Bias / Variance Diagnosis)

In [ ]:
sizes, tr, va = learning_curve(best_model, X_train, y_train, cv=cv, scoring=SCORING,
                               train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1, random_state=RANDOM_STATE)
plt.plot(sizes, tr.mean(1), "o-", label="Train")
plt.plot(sizes, va.mean(1), "o-", label="Validation")
plt.fill_between(sizes, va.mean(1)-va.std(1), va.mean(1)+va.std(1), alpha=.15)
plt.xlabel("Training samples"); plt.ylabel(SCORING); plt.legend(); plt.grid(alpha=.3)
plt.title("Learning curve"); plt.show()

## 8. Feature Importance
Uses native importances/coefficients when available, otherwise permutation importance.

In [ ]:
from sklearn.inspection import permutation_importance

clf = best_model.named_steps["clf"]
if hasattr(clf, "feature_importances_"):
    imp = pd.Series(clf.feature_importances_, index=X.columns)
elif hasattr(clf, "coef_") and clf.coef_.ndim == 2:
    imp = pd.Series(np.abs(clf.coef_[0]), index=X.columns)
else:
    r = permutation_importance(best_model, X_test, y_test, n_repeats=10,
                               random_state=RANDOM_STATE, scoring=SCORING, n_jobs=-1)
    imp = pd.Series(r.importances_mean, index=X.columns)

imp.sort_values().tail(15).plot.barh(figsize=(7,5), color="teal")
plt.title("Top 15 features"); plt.tight_layout(); plt.show()

## 9. Save the Optimized Model

In [ ]:
joblib.dump(best_model, "optimized_model.joblib")
print("Saved -> optimized_model.joblib")

# Reload & sanity check
loaded = joblib.load("optimized_model.joblib")
assert (loaded.predict(X_test) == best_model.predict(X_test)).all()
print("Reload check passed.")

## 10. Summary & Next Steps
- Compare baseline vs. tuned scores above to quantify the gain from optimization.
- Ideas to push further: ensembling (`VotingClassifier`/`StackingClassifier`), XGBoost/LightGBM, class-weighting for imbalance, threshold tuning, calibration, and nested CV for unbiased estimates.